In [ ]:
# # using the EPM code as starting point

# from pathlib import Path
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import re

# # ==========================================================
# # Automated EPM + Fiber Photometry pipeline
# # Flexible to zone numbering & header differences
# # ==========================================================

# # ----------------------------------------------------------
# # Main configuration
# # ----------------------------------------------------------

# main_dir = Path(".").resolve()
# #timepoints = ["Preinduction", "W1", "W2", "W3"]

# #focus on preinduction for now to save time 
# timepoints = ["Preinduction"]
# #timepoints = ["W3"]
# #timepoints = ["Preinduction","W3"]

# # Nose-point zone substrings for 3CT (FLEXIBLE MATCHING)
# nosepoint_substrings = [
#     "Social_interaction",
#     "Novel_interaction",
# ]

# # Fixed colors keyed by inner substring
# zone_colors = {
#     "Social_interaction": "#377EB8",   # blue
#     "Novel_interaction": "#E41A1C",  # red
# }


# video_fps = 50.0
# fp_fps = 50.0

# figures_main_dir = main_dir / "Figures"
# figures_main_dir.mkdir(parents=True, exist_ok=True)

# # ==========================================================
# # Helper functions
# # ==========================================================

# def _mouse_genotype_tag(mouse_id: str):
#     id_clean = mouse_id.strip()
#     if id_clean in {"372", "376", "423", "398", "400", "459"}:
#         return "NE"
#     if id_clean in {"374", "429", "461", "463", "402"}:
#         return "WT"
#     return None


# def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
#     df = pd.read_excel(xl_path, header=None)
#     key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()

#     # FP file
#     match_fp = df[key_col == "fp file"]
#     if match_fp.empty:
#         raise ValueError(f"'FP file' not found in {xl_path.name}")
#     fp_id = str(df.iloc[match_fp.index[0], 1]).strip()

#     # Mouse ID
#     #match_id = df[key_col == "id"]
#     match_id = df[key_col == "id"]
#     if match_id.empty:
#         raise ValueError(f"'ID' row not found in {xl_path.name}")
#     mouse_id = str(df.iloc[match_id.index[0], 1]).strip()

#     return fp_id, mouse_id

# def _extract_condition(eth_path: Path) -> str:
#     """
#     Find the 'Condition' row in the metadata (top of the EthoVision file)
#     and return its value (e.g. Hab, Social, Novel).
#     """
#     raw = pd.read_excel(eth_path, header=None, nrows=50, engine="openpyxl")
#     for i in range(len(raw)):
#         if str(raw.iloc[i, 0]).strip() == "Condition":
#             cond = str(raw.iloc[i, 1]).strip()
#             # Normalise a bit
#             cond_norm = cond.capitalize()
#             if cond_norm in {"Hab", "Social", "Novel"}:
#                 return cond_norm
#             return cond
#     return "Unknown"



# def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str):
#     fp_csv = None
#     for p in fp_folder.glob("*.csv"):
#         if fp_id.lower() in p.name.lower():
#             #fp_csv = p
#             fil = fp_id + '.csv'
#             fp_csv = fp_folder / fil
#             break
#     if fp_csv is None:
#         raise FileNotFoundError(f"No FP CSV found in {fp_folder}")

#     ts_csv = None
#     for p in fp_folder.glob("*.csv"):
#         if "time" in p.name.lower() or "timestamp" in p.name.lower():
#             #ts_csv = p
#             import os
#             directory, filename = os.path.split(p)

#             # Remove leading "._" if present
#             new_filename = filename.lstrip("._") if filename.startswith("._") else filename

#             # Join back together
#             new_path = os.path.join(directory, new_filename)
            
#             ts_csv = new_path
#             break
#     if ts_csv is None:
#         raise FileNotFoundError(f"No timestamp CSV found in {fp_folder}")

#     return fp_csv, ts_csv


# def _coerce_bool_col(series):
#     if series.dtype == bool:
#         return series
#     if pd.api.types.is_numeric_dtype(series):
#         return (series.astype(float) > 0).astype(bool)
#     low = series.astype(str).str.strip().str.lower()
#     return low.isin(["true", "1", "t", "yes", "y"])


# def _get_video_window(ts: pd.DataFrame):
#     """Robust detection of video ON/OFF timestamps."""
#     # Time column
#     tcol = next(c for c in ts.columns if "time" in c.lower())
#     t = pd.to_numeric(ts[tcol], errors="coerce")

#     # Digital channel
#     scol = None
#     for c in ts.columns:
#         if "digital" in c.lower() or "state" in c.lower():
#             scol = c
#             break

#     # No digital info → use full recording
#     if scol is None:
#         return float(t.iloc[0]), float(t.iloc[-1])

#     state = _coerce_bool_col(ts[scol])

#     # If no transitions → full interval
#     if state.sum() == 0 or (~state).sum() == 0:
#         return float(t.iloc[0]), float(t.iloc[-1])

#     # Use first True and first False
#     try:
#         video_start = float(t[state].iloc[0])
#         video_stop = float(t[~state].iloc[0])
#         if video_stop > video_start:
#             return video_start, video_stop
#     except:
#         pass

#     # Fallback
#     return float(t.iloc[0]), float(t.iloc[-1])


# def _build_time_vector_from_fp(fp_df):
#     if "SystemTimestamp" in fp_df.columns:
#         ts = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce")
#         if ts.notna().sum() > len(ts) * 0.8:
#             return ts.to_numpy()
#     return np.arange(len(fp_df)) / fp_fps


# def _snap_down_index(t, target):
#     return max(0, np.searchsorted(t, target, side="right") - 1)


# def _trim_fp_to_window(fp_df, start, stop):
#     tt = _build_time_vector_from_fp(fp_df)
#     if len(tt) != len(fp_df):
#         n = min(len(tt), len(fp_df))
#         tt = tt[:n]
#         fp_df = fp_df.iloc[:n]

#     i0 = _snap_down_index(tt, start)
#     i1 = _snap_down_index(tt, stop)
#     out = fp_df.iloc[i0:i1+1].copy()
#     out["Time_video"] = tt[i0:i1+1] - tt[i0]
#     return out


# def _detect_header_row(eth_path: Path):
#     preview = pd.read_excel(eth_path, header=None, nrows=50)

#     firstcell = str(preview.iloc[0, 0])
#     if "number of header lines" in firstcell.lower():
#         m = re.findall(r"\d+", firstcell)
#         if m:
#             return int(m[0])

#     col0 = preview.iloc[:, 0].astype(str)
#     hits = col0[col0.str.contains("Trial time", case=False, na=False)]
#     if not hits.empty:
#         return hits.index[0]

#     raise ValueError(f"Cannot detect header row in {eth_path.name}")


# def _load_ethovision_sheet_with_targets(eth_path: Path, video_fps: float):
#     header_row = _detect_header_row(eth_path)
#     eth = pd.read_excel(eth_path, header=header_row).dropna(axis=1, how="all")

#     # Time_s
#     tcol = next((c for c in eth.columns if "time" in c.lower()), None)
#     if tcol:
#         eth["Time_s"] = pd.to_numeric(eth[tcol], errors="coerce")
#     else:
#         eth["Time_s"] = np.arange(len(eth)) / video_fps

#     # Flexible matching of nose-point zone columns
#     matched_cols = []
#     for sub in nosepoint_substrings:
#         for col in eth.columns:
#             if sub in str(col):
#                 matched_cols.append(col)
#                 break

#     if not matched_cols:
#         raise ValueError(f"No nose-point zones found in {eth_path.name}")

#     return eth[["Time_s"] + matched_cols].copy(), matched_cols


# def _ethogram_segments(eth, beh_cols):
#     segments = []
#     for beh in beh_cols:
#         mask = eth[beh] == 1
#         if mask.sum() == 0:
#             continue
#         diff = mask.astype(int).diff().fillna(0)
#         starts = eth.loc[diff == 1, "Time_s"]
#         ends = eth.loc[diff == -1, "Time_s"]

#         if mask.iloc[0]:
#             starts = pd.concat([pd.Series([eth["Time_s"].iloc[0]]), starts])
#         if mask.iloc[-1]:
#             ends = pd.concat([ends, pd.Series([eth["Time_s"].iloc[-1]])])

#         for s, e in zip(starts, ends):
#             segments.append(
#                 {"Behavior": beh, "Start_time_s": float(s), "End_time_s": float(e),
#                  "Duration_s": float(e - s)}
#             )
#     return pd.DataFrame(segments)


# def _guess_zone_color(colname):
#     for sub, color in zone_colors.items():
#         if sub in colname:
#             return color
#     return "black"


# def _plot_trial(fp_trim, eth_timeline, beh_cols, out_png, title_prefix):
#     fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True,
#                              gridspec_kw={"height_ratios": [2, 2, 1]})

#     chan = "G0" if "G0" in fp_trim.columns else next(
#         c for c in fp_trim.columns
#         if pd.api.types.is_numeric_dtype(fp_trim[c])
#         and c not in {"LedState", "SystemTimestamp", "Time_video", "Timestamp"}
#     )

#     # LED 1 – isosbestic
#     fp1 = fp_trim[fp_trim["LedState"] == 1]
#     axes[0].plot(fp1["Time_video"], fp1[chan], linewidth=0.8, color="gray")
#     axes[0].set_title(f"{title_prefix}: LED 1 (isosbestic)")
#     axes[0].grid(True)

#     # LED 2 – GCaMP
#     fp2 = fp_trim[fp_trim["LedState"] == 2]
#     axes[1].plot(fp2["Time_video"], fp2[chan], linewidth=0.8, color="green")
#     axes[1].set_title(f"{title_prefix}: LED 2 (GCaMP)")
#     axes[1].grid(True)

#     # Ethogram
#     for i, beh in enumerate(beh_cols):
#         color = _guess_zone_color(beh)
#         subset = eth_timeline[eth_timeline["Behavior"] == beh]
#         for _, row in subset.iterrows():
#             axes[2].barh(i, row["Duration_s"], left=row["Start_time_s"],
#                          height=0.6, color=color)

#     axes[2].set_yticks(range(len(beh_cols)))
#     axes[2].set_yticklabels(beh_cols)
#     axes[2].invert_yaxis()
#     axes[2].set_xlabel("Time (s)")
#     axes[2].set_title("Time in zone (nose-point)")

#     plt.tight_layout()
#     plt.savefig(out_png, dpi=200)
#     plt.close(fig)


# def _extract_led_traces(fp_trim, chan):
#     iso = fp_trim.loc[fp_trim["LedState"] == 1, ["Time_video", chan]].rename(
#         columns={chan: f"{chan}_iso415"})
#     gcamp = fp_trim.loc[fp_trim["LedState"] == 2, ["Time_video", chan]].rename(
#         columns={chan: f"{chan}_gcamp470"})
#     return iso.reset_index(drop=True), gcamp.reset_index(drop=True)

# # ==========================================================
# # Main loop with week/timepoint added
# # ==========================================================

# fp_traces = {}

# for tp in timepoints:
#     print(f"\n=== Processing timepoint: {tp} ===")
#     tp_dir = main_dir / tp
#     export_dir = tp_dir / "Export files"

#     fig_dir_tp = figures_main_dir / tp
#     fig_dir_raw = fig_dir_tp / "Raw_traces_ethograms"
#     fig_dir_raw.mkdir(parents=True, exist_ok=True)

#     trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))
#     if not trial_files:
#         print(f"[WARN] No Raw data-*.xlsx files in {export_dir}")
#         continue

#     for eth_path in trial_files:
#         try:
#             # Condition (Hab / Social / Novel)
#             condition = _extract_condition(eth_path)
#             fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)
#             geno = _mouse_genotype_tag(mouse_id)

#             fp_folder = tp_dir / fp_id
#             fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)

#             # Load FP
#             fp = pd.read_csv(fp_csv, encoding="utf-8")
#             fp = fp[fp["LedState"].isin([1, 2])]

#             # Align timestamps
#             ts = pd.read_csv(ts_csv, encoding="utf-8")
#             video_start, video_stop = _get_video_window(ts)
#             fp_trim = _trim_fp_to_window(fp, video_start, video_stop)

#             # Detect channel
#             chan = "G0" if "G0" in fp_trim.columns else next(
#                 c for c in fp_trim.columns
#                 if pd.api.types.is_numeric_dtype(fp_trim[c])
#                 and c not in {"LedState", "SystemTimestamp", "Time_video", "Timestamp"}
#             )

#             iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)

#             trial_stub = eth_path.stem.replace("Raw data-", "").replace(" ", "_")
#             mouse_label = f"ID{mouse_id}" + (f"_{geno}" if geno else "")
#             file_stub = f"{mouse_label}__{fp_id}__{trial_stub}"

#             # Load EthoVision
#             eth, matched_cols = _load_ethovision_sheet_with_targets(eth_path, video_fps)
#             timeline = _ethogram_segments(eth, matched_cols)

#             # Store all info in fp_traces, now including the 'week' field
#             fp_traces[f"{tp}__{file_stub}"] = {
#                 "iso": iso_df,
#                 "gcamp": gcamp_df,
#                 "mouse_id": mouse_id,
#                 "genotype": geno,
#                 "timeline": timeline,
#                 "beh_cols": matched_cols,
#                 "condition": condition,
#                 "week": tp  # <--- NEW: store the timepoint/week
#             }

#             # Figure directory: by week and condition
#             fig_dir_tp_cond = figures_main_dir / tp / condition / "Raw_traces_ethograms"
#             fig_dir_tp_cond.mkdir(parents=True, exist_ok=True)

#             # Plot trial
#             out_png = fig_dir_tp_cond / f"{file_stub}_zones.png"
#             _plot_trial(fp_trim, timeline, matched_cols, out_png,
#                         title_prefix=f"{mouse_label} | {fp_id} | {trial_stub}")

#             print(f"✔ {eth_path.name} → {out_png.name}")

#         except Exception as e:
#             print(f"[WARN] [{tp}] {eth_path.name}: {e}")



=== Processing timepoint: Preinduction ===
✔ Raw data-3CT_Preinduction-Trial    17.xlsx → ID423_NE__3CT_FP_11_1__3CT_Preinduction-Trial____17_zones.png


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import re

# ==========================================================
# CONFIG
# ==========================================================

main_dir = Path(".").resolve()
timepoints = ["Preinduction"]

video_fps = 50.0
fp_fps = 50.0

figures_main_dir = main_dir / "Figures"
figures_main_dir.mkdir(parents=True, exist_ok=True)

# ==========================================================
# VIDEO OVERLAY FUNCTION
# ==========================================================

def create_fp_overlay_video(video_path, fp_df, output_path, window=5):
    """
    Overlay FP trace onto video.

    window = seconds of trace shown before current time
    """

    if not video_path.exists():
        print(f"[WARN] No video found: {video_path}")
        return

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    times = fp_df["Time_video"].values
    signal = fp_df.iloc[:, 1].values

    ymin, ymax = np.nanmin(signal), np.nanmax(signal)

    frame_idx = 0

    print(f"🎬 Creating overlay video: {output_path.name}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        t = frame_idx / fps

        idx = np.searchsorted(times, t)
        idx = np.clip(idx, 1, len(times)-1)

        # sliding window
        t_start = max(0, t - window)

        mask = (times >= t_start) & (times <= t)

        fig, ax = plt.subplots(figsize=(4, 2))

        ax.plot(times[mask], signal[mask], color="green", linewidth=1)
        ax.axvline(t, color="red", linewidth=1)

        ax.set_xlim(t_start, t)
        ax.set_ylim(ymin, ymax)

        ax.axis("off")

        fig.canvas.draw()

        plot_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        plot_img = plot_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        plt.close(fig)

        plot_img = cv2.resize(plot_img, (int(width * 0.35), int(height * 0.25)))

        h, w, _ = plot_img.shape

        # bottom-left overlay
        frame[-h:, :w] = plot_img

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

    print(f"✔ Saved video: {output_path}")


# ==========================================================
# HELPER FUNCTIONS (UNCHANGED CORE)
# ==========================================================

def _read_export_and_get_fp_and_mouse_id(xl_path: Path):
    df = pd.read_excel(xl_path, header=None)
    key_col = df.iloc[:, 0].astype(str).str.strip().str.lower()

    fp_id = str(df.iloc[key_col[key_col == "fp file"].index[0], 1]).strip()
    mouse_id = str(df.iloc[key_col[key_col == "id"].index[0], 1]).strip()

    return fp_id, mouse_id


def _find_fp_and_timestamp_files(fp_folder: Path, fp_id: str):
    fp_csv = fp_folder / f"{fp_id}.csv"

    ts_csv = None
    for p in fp_folder.glob("*.csv"):
        if "time" in p.name.lower():
            ts_csv = p
            break

    if ts_csv is None:
        raise FileNotFoundError("No timestamp CSV found")

    return fp_csv, ts_csv


def _get_video_window(ts):
    tcol = next(c for c in ts.columns if "time" in c.lower())
    t = pd.to_numeric(ts[tcol], errors="coerce")

    return float(t.iloc[0]), float(t.iloc[-1])


def _build_time_vector_from_fp(fp_df):
    if "SystemTimestamp" in fp_df.columns:
        ts = pd.to_numeric(fp_df["SystemTimestamp"], errors="coerce")
        return ts.to_numpy()
    return np.arange(len(fp_df)) / fp_fps


def _trim_fp_to_window(fp_df, start, stop):
    tt = _build_time_vector_from_fp(fp_df)

    i0 = np.searchsorted(tt, start)
    i1 = np.searchsorted(tt, stop)

    out = fp_df.iloc[i0:i1].copy()
    out["Time_video"] = tt[i0:i1] - tt[i0]

    return out


def _extract_led_traces(fp_trim, chan):
    iso = fp_trim.loc[fp_trim["LedState"] == 1, ["Time_video", chan]]
    gcamp = fp_trim.loc[fp_trim["LedState"] == 2, ["Time_video", chan]]
    return iso.reset_index(drop=True), gcamp.reset_index(drop=True)


# ==========================================================
# MAIN LOOP
# ==========================================================

for tp in timepoints:
    print(f"\n=== Processing {tp} ===")

    tp_dir = main_dir / tp
    export_dir = tp_dir / "Export files"

    trial_files = sorted(export_dir.glob("Raw data-*.xlsx"))

    for eth_path in trial_files:
        try:
            fp_id, mouse_id = _read_export_and_get_fp_and_mouse_id(eth_path)

            fp_folder = tp_dir / fp_id
            fp_csv, ts_csv = _find_fp_and_timestamp_files(fp_folder, fp_id)

            # --- LOAD FP ---
            fp = pd.read_csv(fp_csv)
            fp = fp[fp["LedState"].isin([1, 2])]

            # --- ALIGN ---
            ts = pd.read_csv(ts_csv)
            start, stop = _get_video_window(ts)
            fp_trim = _trim_fp_to_window(fp, start, stop)

            # detect channel
            chan = next(
                c for c in fp_trim.columns
                if pd.api.types.is_numeric_dtype(fp_trim[c])
                and c not in {"LedState", "SystemTimestamp", "Time_video"}
            )

            iso_df, gcamp_df = _extract_led_traces(fp_trim, chan)

            # --- FIND VIDEO ---
            video_path = None
            for ext in ["*.mp4", "*.avi", "*.mov"]:
                vids = list(fp_folder.glob(ext))
                if vids:
                    video_path = vids[0]
                    break

            if video_path is None:
                print(f"[WARN] No video in {fp_folder}")
                continue

            # --- OUTPUT ---
            out_dir = figures_main_dir / tp / "Overlay_videos"
            out_dir.mkdir(parents=True, exist_ok=True)

            out_video = out_dir / f"{fp_id}_overlay.mp4"

            # --- CREATE VIDEO ---
            create_fp_overlay_video(
                video_path,
                gcamp_df,
                out_video,
                window=5
            )

        except Exception as e:
            print(f"[ERROR] {eth_path.name}: {e}")


=== Processing Preinduction ===


[ WARN:0@575.717] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 204463.746814 ms
[ WARN:0@575.717] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 204465.803118 ms
[ WARN:0@575.717] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 204465.862952 ms
[ WARN:0@575.717] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 204465.872869 ms
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x13d0bc6d0] moov atom not found


🎬 Creating overlay video: 3CT_FP_11_1_overlay.mp4


/var/folders/3y/jmv30wb16ss901m1yrdysjh80000gq/T/ipykernel_47111/3403272448.py:78: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(t_start, t)


✔ Saved video: /Users/annastuckert/Documents/GitHub/FP_behaviour_analysis/3CT_Conbined_manual_annotation/write_FP_video/Figures/Preinduction/Overlay_videos/3CT_FP_11_1_overlay.mp4
